# Image Caption Generator — COCO Training Notebook

Trains the same architecture as `train_colab.ipynb` (ResNet encoder + Bahdanau attention + LSTM decoder), but on a subset of **MS COCO** instead of Flickr8k.

**Why**: a model trained on Flickr8k (8,000 images, mostly people/animals in action) tested well on Flickr8k-style photos but hallucinated a person on a people-free storefront photo -- see the README's Limitations section. COCO (330k images, 80 object categories, far more scene diversity) is the standard fix for that kind of narrow-distribution failure. This notebook trains on a subset of COCO using the **Karpathy split** -- the exact train/val/test partition used by the original *Show, Attend and Tell* paper and most published captioning baselines, so results here are comparable to literature, not just internally consistent.

**Before running:** Runtime → Change runtime type → GPU (T4 is fine).

**Time budget and resilience**: this is a long-running job by design, sized to survive free-tier Colab's unpredictable session limits rather than assume a single uninterrupted sitting:
- Downloading the image subset takes roughly 30-40 minutes the *first* time (network-bound, ~25 images/sec with 100 parallel downloads). Every session after that restores from a Drive-cached archive instead (section 3), which is much faster.
- Training checkpoints resumable state after *every* epoch, backed up to Drive (section 4). If a session disconnects, reopening this notebook and re-running the cells continues training from the last completed epoch instead of starting over.
- Net effect: expect to run this notebook across **multiple Colab sessions**, not necessarily one. That's the intended workflow, not a failure state.

Steps:
1. Clone the repo and install dependencies
2. Mount Google Drive early (so image caching and mid-training backups work)
3. Get a COCO subset via the Karpathy split (Hugging Face `yerevann/coco-karpathy`) -- restores from a Drive-cached archive if a previous session already downloaded one
4. Train, resumable across sessions via a Drive-backed checkpoint
5. Evaluate (BLEU / METEOR / CIDEr) -- compare directly against the Flickr8k run's numbers in the README
6. Re-run the same out-of-distribution storefront-style test to check whether hallucination improved

## 1. Get the code onto Colab

In [ ]:
# Repo is private -- clone with a token (generate one at
# github.com -> Settings -> Developer settings -> Personal access tokens,
# 'repo' scope; revoke it once the clone succeeds, it's not needed again)
!git clone https://<your-github-token>@github.com/SovanDaraP02/image-caption-generator.git
%cd image-caption-generator

!pwd
!ls

In [ ]:
!pip install -q -e .
!pip install -q datasets

In [ ]:
import os
import sys
# pip install -e . updates site-packages on disk, but this already-running
# kernel cached its module search path at startup, so it won't see the
# newly-installed package until the kernel restarts. Rather than restart
# (which would lose any state from cells run before this point), just add
# the source directory to sys.path directly -- every cell below that
# imports caption_generator relies on this having run first.
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (go enable a GPU runtime!)")

## 2. Mount Google Drive early (for mid-training backups)

In [ ]:
# Mounted up front (not just at the end) so the download step (section 3)
# can cache images to Drive, and the training loop (section 4) can back
# up every new best checkpoint as it happens -- this run is long enough
# that losing either to a disconnect partway through would be costly.
# Not fatal if it fails: everything still proceeds, just without those
# safety nets.
DRIVE_AVAILABLE = False
DRIVE_DIR = "/content/drive/MyDrive/image-caption-generator"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    os.makedirs(DRIVE_DIR, exist_ok=True)
    DRIVE_AVAILABLE = True
    print("Drive mounted -- image caching and mid-training backups enabled.")
except Exception as e:
    print(f"Drive mount failed ({e}) -- continuing without Drive caching/backups.")

## 3. Download a COCO subset (Karpathy split)

In [ ]:
# Verified working source as of writing: yerevann/coco-karpathy on the Hub.
# It exposes the Karpathy partition as four SEPARATE loadable HF splits
# (train / restval / validation / test) -- not one stream with a 'split'
# column to filter -- confirmed by inspecting each directly below. It's
# text/metadata only (captions + an image URL per row); images are
# fetched separately from the official COCO image server in the next
# cell. If this mirror ever goes away, load the alternative and re-run
# this cell to see its actual split names/fields before adjusting
# collect_split_examples below.
from datasets import load_dataset

for split in ["train", "restval", "validation", "test"]:
    ds = load_dataset("yerevann/coco-karpathy", split=split, streaming=True)
    first = next(iter(ds))
    print(f"{split:12s} -> internal split label: {first['split']:8s}  e.g. {first['filename']}")

In [ ]:
# Subset sizes -- tune these down if you want a faster run, or up for even
# more diversity (full Karpathy train split is ~113k images). 50k/3k/3k
# keeps the download to roughly half an hour at ~25 img/s.
N_TRAIN = 50_000
N_VAL = 3_000
N_TEST = 3_000

IMAGE_DIR = "data/coco/Images"
import os
os.makedirs(IMAGE_DIR, exist_ok=True)

In [ ]:
import shutil
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests


def collect_split_examples(hf_splits, n):
    """hf_splits: list of HF split names to pull from, in order, stopping
    once n examples are collected. For 'train' we pass ["train", "restval"]
    since 'restval' is COCO's extra pool the Karpathy split adds to train
    in most published setups -- pulled in automatically once the plain
    'train' rows (~82k) run out, so N_TRAIN can exceed that."""
    examples = []
    for hf_split in hf_splits:
        if len(examples) >= n:
            break
        stream = load_dataset("yerevann/coco-karpathy", split=hf_split, streaming=True)
        for ex in stream:
            examples.append(ex)
            if len(examples) >= n:
                break
    return examples


IMAGE_CACHE_TAR_DRIVE = f"{DRIVE_DIR}/coco_images_cache.tar"
IMAGE_CACHE_TAR_LOCAL = "/content/coco_images_cache.tar"


def try_restore_image_cache():
    """If a previous session already archived its downloaded images to
    Drive, extract that instead of re-downloading ~50k individual files
    from COCO's server. That re-download is the single biggest cost of
    resuming after a disconnect (network-bound, ~35 min for the default
    subset size) -- this cuts it down to however long extracting one big
    archive from Drive takes, typically much less. download_images below
    also skips any file that already exists locally, so even a partial
    or slightly-stale cache just means downloading the few images it's
    missing, not starting over."""
    if not (DRIVE_AVAILABLE and os.path.exists(IMAGE_CACHE_TAR_DRIVE)):
        return False
    size_gb = os.path.getsize(IMAGE_CACHE_TAR_DRIVE) / 1e9
    print(f"Found cached images archive on Drive ({size_gb:.1f} GB) -- restoring instead of re-downloading...")
    start = time.time()
    subprocess.run(["tar", "-xf", IMAGE_CACHE_TAR_DRIVE, "-C", "."], check=True)
    print(f"Restored in {time.time() - start:.0f}s")
    return True


def save_image_cache_to_drive():
    """Archives IMAGE_DIR to Drive so a future session's
    try_restore_image_cache() can skip re-downloading. Best-effort --
    failure here just means the next session re-downloads instead."""
    if not DRIVE_AVAILABLE:
        return
    try:
        print("Archiving downloaded images to Drive for future sessions...")
        start = time.time()
        subprocess.run(["tar", "-cf", IMAGE_CACHE_TAR_LOCAL, IMAGE_DIR], check=True)
        shutil.copy(IMAGE_CACHE_TAR_LOCAL, IMAGE_CACHE_TAR_DRIVE)
        print(f"Cached to Drive in {time.time() - start:.0f}s -- future sessions' download step will be much faster.")
    except Exception as e:
        print(f"Image cache backup failed ({e}) -- not fatal, next session just re-downloads.")


def download_images(examples, max_workers=100, max_retries=2):
    """Downloads to IMAGE_DIR, returns (image_filename, caption) pairs --
    one pair per caption, matching Flickr8k's 5-captions-per-image density.
    Skips any file that already exists (e.g. restored from the Drive
    cache above), so this is safe and fast to call even when nothing
    actually needs downloading."""
    def fetch(ex):
        path = os.path.join(IMAGE_DIR, ex["filename"])
        if os.path.exists(path):
            return ex, True
        for attempt in range(max_retries + 1):
            try:
                r = requests.get(ex["url"], timeout=10)
                r.raise_for_status()
                with open(path, "wb") as f:
                    f.write(r.content)
                return ex, True
            except Exception:
                if attempt == max_retries:
                    return ex, False
                time.sleep(0.5)

    pairs = []
    failed = 0
    start = time.time()
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = [pool.submit(fetch, ex) for ex in examples]
        for i, future in enumerate(as_completed(futures), 1):
            ex, ok = future.result()
            if ok:
                for cap in ex["sentences"]:
                    pairs.append((ex["filename"], cap))
            else:
                failed += 1
            if i % 2000 == 0:
                elapsed = time.time() - start
                print(f"  {i}/{len(examples)} images ({i/elapsed:.1f} img/s), {failed} failed")
    print(f"Done: {len(examples) - failed}/{len(examples)} images, {len(pairs)} (image, caption) pairs, {failed} failed")
    return pairs


cache_restored = try_restore_image_cache()

print("Collecting split assignments...")
train_examples = collect_split_examples(["train", "restval"], N_TRAIN)
val_examples = collect_split_examples(["validation"], N_VAL)
test_examples = collect_split_examples(["test"], N_TEST)
print(f"train: {len(train_examples)}  val: {len(val_examples)}  test: {len(test_examples)}")

print("Downloading train images..." if not cache_restored else "Verifying train images against restored cache...")
TRAIN_PAIRS = download_images(train_examples)
print("Downloading val images..." if not cache_restored else "Verifying val images against restored cache...")
VAL_PAIRS = download_images(val_examples)
print("Downloading test images..." if not cache_restored else "Verifying test images against restored cache...")
TEST_PAIRS = download_images(test_examples)

if not cache_restored:
    save_image_cache_to_drive()

## 4. Train (resumable across disconnects)

Same `train_one_epoch`/`validate` as the Flickr8k notebook, plus one
difference that matters a lot at this dataset size: free-tier Colab
sessions can be reclaimed with little warning, and one epoch over the
full 250k-pair training set can take much longer than a single session
lasts. Rather than losing that progress, this cell saves full resumable
state (model + optimizer + scheduler + epoch number) after *every*
epoch to `latest_checkpoint_coco.pth`, backed up to Drive -- and at the
top of the cell, automatically picks that back up if it exists, instead
of starting over. Practically: if your session gets cut off, just
re-run cells 1-3 (which re-downloads images -- unavoidable, the VM disk
doesn't survive a disconnect) then re-run this cell; it'll skip
straight to the epoch after the last one that finished.

In [ ]:
import os
import shutil

import torch.nn as nn
from torch.utils.data import DataLoader

from caption_generator.data.dataset import ImageCaptionDataset, collate_fn
from caption_generator.data.vocabulary import Vocabulary
from caption_generator.models.decoder import DecoderWithAttention
from caption_generator.models.encoder import EncoderCNN
from caption_generator.train import train_one_epoch, validate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

CHECKPOINT_PATH = "best_checkpoint_coco.pth"       # encoder+decoder+vocab only -- used by evaluate/app
LATEST_PATH = "latest_checkpoint_coco.pth"          # full resumable state, saved every epoch
DRIVE_DIR = "/content/drive/MyDrive/image-caption-generator"

TRAIN_CAPTIONS_RAW = [cap for _, cap in TRAIN_PAIRS]
vocab = Vocabulary(min_word_freq=5).build(TRAIN_CAPTIONS_RAW)
pad_idx = vocab.word2idx[vocab.PAD_TOKEN]
print(f"Vocab size: {len(vocab)}")

train_dataset = ImageCaptionDataset(IMAGE_DIR, TRAIN_PAIRS, vocab, split="train")
val_dataset = ImageCaptionDataset(IMAGE_DIR, VAL_PAIRS, vocab, split="val")

# Bigger batch than the Flickr8k notebook (64 vs 32): the encoder is
# frozen so memory footprint is modest, and observed GPU RAM usage at
# batch_size=32 was only ~1.5/15GB on a T4 -- plenty of headroom to trade
# for fewer, larger steps per epoch and meaningfully less wall-clock time.
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           collate_fn=lambda b: collate_fn(b, pad_idx), num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=lambda b: collate_fn(b, pad_idx), num_workers=2)

encoder = EncoderCNN(fine_tune=False).to(device)
decoder = DecoderWithAttention(vocab_size=len(vocab)).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
optimizer = torch.optim.Adam(decoder.parameters(), lr=4e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

NUM_EPOCHS = 10  # 8x more data per epoch than Flickr8k -- resumable, so this is a target, not a single-sitting requirement
EARLY_STOP_PATIENCE = 3
best_val_loss = float("inf")
epochs_without_improvement = 0
start_epoch = 1

# --- Resume support: pick up where a previous (disconnected) session left off ---
resume_source = None
if os.path.exists(LATEST_PATH):
    resume_source = LATEST_PATH
elif DRIVE_AVAILABLE and os.path.exists(f"{DRIVE_DIR}/{LATEST_PATH}"):
    resume_source = f"{DRIVE_DIR}/{LATEST_PATH}"

if resume_source:
    print(f"Found a previous run's checkpoint at {resume_source} -- resuming instead of starting over.")
    ckpt = torch.load(resume_source, map_location=device)
    encoder.load_state_dict(ckpt["encoder_state"])
    decoder.load_state_dict(ckpt["decoder_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    vocab.word2idx = ckpt["vocab_word2idx"]
    vocab.idx2word = ckpt["vocab_idx2word"]
    pad_idx = vocab.word2idx[vocab.PAD_TOKEN]
    best_val_loss = ckpt["best_val_loss"]
    epochs_without_improvement = ckpt["epochs_without_improvement"]
    start_epoch = ckpt["epoch"] + 1
    print(f"Resuming from epoch {start_epoch}, best_val_loss so far = {best_val_loss:.4f}")
else:
    print("No previous checkpoint found -- starting fresh.")

for epoch in range(start_epoch, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(encoder, decoder, train_loader, optimizer, criterion, device, pad_idx)
    val_loss = validate(encoder, decoder, val_loader, criterion, device)
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch}/{NUM_EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  lr={current_lr:.2e}")

    improved = val_loss < best_val_loss
    if improved:
        best_val_loss = val_loss
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    # Always save resumable "latest" state, improved or not -- this is
    # what lets a fresh session continue from here instead of restarting.
    torch.save({
        "encoder_state": encoder.state_dict(),
        "decoder_state": decoder.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "vocab_word2idx": vocab.word2idx,
        "vocab_idx2word": vocab.idx2word,
        "epoch": epoch,
        "best_val_loss": best_val_loss,
        "epochs_without_improvement": epochs_without_improvement,
    }, LATEST_PATH)

    if improved:
        torch.save({
            "encoder_state": encoder.state_dict(),
            "decoder_state": decoder.state_dict(),
            "vocab_word2idx": vocab.word2idx,
            "vocab_idx2word": vocab.idx2word,
        }, CHECKPOINT_PATH)
        print(f"  -> saved new best checkpoint (val_loss={val_loss:.4f})")

    if DRIVE_AVAILABLE:
        try:
            os.makedirs(DRIVE_DIR, exist_ok=True)
            shutil.copy(LATEST_PATH, f"{DRIVE_DIR}/{LATEST_PATH}")
            if improved:
                shutil.copy(CHECKPOINT_PATH, f"{DRIVE_DIR}/{CHECKPOINT_PATH}")
            print(f"  -> backed up to Drive (latest{' + best' if improved else ''})")
        except Exception as e:
            print(f"  -> Drive backup failed this epoch ({e}), will retry next epoch")

    if epochs_without_improvement >= EARLY_STOP_PATIENCE:
        print(f"No val_loss improvement for {EARLY_STOP_PATIENCE} epochs -- stopping early.")
        break

## 5. Evaluate on the test split (BLEU / METEOR / CIDEr)

In [ ]:
!pip install -q pycocoevalcap

from collections import defaultdict

from caption_generator.evaluate import evaluate

test_pairs_by_image = defaultdict(list)
for fname, cap in TEST_PAIRS:
    test_pairs_by_image[fname].append(cap)

scores = evaluate(CHECKPOINT_PATH, dict(test_pairs_by_image), IMAGE_DIR, device=device)
for metric, value in scores.items():
    print(f"{metric}: {value:.4f}")

print("\nCompare against the Flickr8k run's numbers in README.md's Results table.")

## 6. Re-test the hallucination case

Upload the same kind of people-free photo that the Flickr8k model got
wrong (or any test image) via the Colab file browser, then run this cell
against it to see whether the COCO-trained model still hallucinates a
person.

In [ ]:
import torchvision.transforms as T
from PIL import Image

from caption_generator.models.caption_model import CaptionModel

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
vocab = Vocabulary()
vocab.word2idx = checkpoint["vocab_word2idx"]
vocab.idx2word = checkpoint["vocab_idx2word"]

encoder = EncoderCNN(fine_tune=False)
encoder.load_state_dict(checkpoint["encoder_state"])
decoder = DecoderWithAttention(vocab_size=len(vocab))
decoder.load_state_dict(checkpoint["decoder_state"])
model = CaptionModel(encoder, decoder, vocab, device=device)

transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# CHANGE THIS to an uploaded file's path (Colab file browser -> upload -> copy path)
TEST_IMAGE_PATH = "data/coco/Images/" + TEST_PAIRS[0][0]

image = Image.open(TEST_IMAGE_PATH).convert("RGB")
image_tensor = transform(image).unsqueeze(0)

caption_beam = model.generate_beam(image_tensor, beam_width=3)
print(f"Caption: {caption_beam}")

import matplotlib.pyplot as plt
plt.imshow(image)
plt.title(caption_beam)
plt.axis("off")
plt.show()

## 7. Final save to Google Drive (in case section 2's mount wasn't available, or you want a final confirmed copy)

In [ ]:
from google.colab import files

try:
    if not DRIVE_AVAILABLE:
        from google.colab import drive
        drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/image-caption-generator
    !cp {CHECKPOINT_PATH} /content/drive/MyDrive/image-caption-generator/
    print(f"Saved to Google Drive: MyDrive/image-caption-generator/{CHECKPOINT_PATH}")
except Exception as e:
    print(f"Drive save failed ({e}); falling back to direct download instead.")
    files.download(CHECKPOINT_PATH)